# **Thames Valley Crime Data Analysis**
## EDA: Exploratory Data Analysis
---

### Objectives

- Use the cleaned dataset from `01_ETL.ipynb` to answer Key Analytical Questions 1-3 (see below).
- Describe the basic characteristics of the data: averages, spread, distributions, outliers and correlations.
- Find which crime types are most common, how crime differs between the 13 areas and whether crime has risen or fallen over the 36 months.
- Explore the seasonal pattern (if any), ready for the statistical test in `03_statistical_testing.ipynb`.
- Decide which charts to rebuild for the Streamlit dashboard.

### Planned Steps

1. Load and check the cleaned data
2. Descriptive statistics, distributions, outliers and correlations
3. Question 1 - Which crimes are the most common?
4. Question 2 - How does crime differ between the 13 areas?
5. Question 3 - Has crime risen or fallen over the 36 months?
6. Question 4 - Is there a seasonal pattern? (charts only)
7. Conclusions
8. Decide on charts to rebuild for the dashboard, and next steps

### Inputs

`../data/clean/crime_summary.csv` - Crime counts per month, season, area and crime type.

The summary file is used rather than `crime_clean.csv`, because all the questions here are about **how many** crimes there are. It is much smaller (0.3 MB), so it is quicker to load.

### Key Questions explored in this notebook

1. Which reported crime types are the most common across the Thames Valley?
2. How does crime differ between the 13 areas?
3. Has the number of crimes risen or fallen between October 2023 and September 2026?

**Explored here with charts; the statistical test is in `03_statistical_testing.ipynb`**

4. Is there a seasonal pattern, and if so is the difference between summer and winter statistically different?

**Will be answered in `04_modelling.ipynb`**

5. Can monthly crime counts be predicted for an area and crime type, if so which model does it best?

### Outputs

- Summary statistics, distributions, outliers and correlations.
- Charts and findings for Key Questions 1-3.
- Seasonal charts for Key Question 4.

### Additional Comments

- Cells in this notebook run top to bottom.
- All figures are **counts**, not **rates**. Larger areas naturally have more crime (see Limitations in the README).
---

### Setup

Import required libraries:

- `pandas` - to load and group the data.
- `matplotlib` and `seaborn` - to draw charts.

In [1]:
# Import required libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully")

Libraries imported successfully


---
### 1. Load and check the cleaned data

Here we will load and check the summary data from `01_ETL.ipynb` to make sure nothing was lost or changed when it was saved.

In [4]:
summary = pd.read_csv("../data/clean/crime_summary.csv")
summary.head()


,Month,Season,Area,Crime type,Count
0,2023-10-01,Autumn,Bracknell Forest,Anti-social behaviour,81
1,2023-10-01,Autumn,Buckinghamshire,Anti-social behaviour,306
2,2023-10-01,Autumn,Cherwell,Anti-social behaviour,90
3,2023-10-01,Autumn,Milton Keynes,Anti-social behaviour,171
4,2023-10-01,Autumn,Oxford,Anti-social behaviour,125


In [ ]:
# Check the data type of each column
summary.dtypes

Month         object
Season        object
Area          object
Crime type    object
Count          int64
dtype: object

***Month*** shows as `object`, which means it has been read as text. This is because a CSV stores everything as text, so the date format from the ETL notebook was lost when the file was saved. We need to convert it back to a date using `pd.to_datetime`.

***Season***, ***Area***, and ***Crime type*** are text, ***Count*** is a whole number (`int64`) - so these are correct.

In [8]:
# Convert month back to a date
summary['Month'] = pd.to_datetime(summary['Month'])

# Check the conversion
summary.dtypes

Month         datetime64[ns]
Season                object
Area                  object
Crime type            object
Count                  int64
dtype: object

***Month*** is now showing in the correct format, and ready to use.

Quick checks to make sure the loaded file matches what was exported in `01_ETL.ipynb`:

In [ ]:
# Check number of rows (should be 6,552)
print("Rows:", len(summary))

# Check the number of columns (should be 5)
print("Columns:", len(summary.columns))

# Check the column names (should be Month, Season, Area, Crime type and Count)
print("Column Names:", summary.columns.tolist())

# Add the count column for each row (should total 579,772)
print("Total crimes:", summary["Count"].sum())

# Check number of months, areas and crime types (should be 36, 13 and 14)
print("Number of Months:", summary["Month"].nunique())
print("Number of Areas:", summary["Area"].nunique())
print("Number of Crime Types:", summary["Crime type"].nunique())

# Check for any missing values in the dataset
print("Total Missing Values:", summary.isnull().sum().sum())


Rows 6552
Columns 5
Column Names: ['Month', 'Season', 'Area', 'Crime type', 'Count']
Total crimes: 579772
Number of Months: 36
Number of Areas: 13
Number of Crime Types: 14
Total Missing Values: 0


### 2. Descriptive statistics, distributions, outliers and correlations

#### Summary Statistics

**Mean**: the average (add up all the values and divide by how many there are).

**Median**: the middle value when all the values are put in order. It is not pulled up or down by a few very big or small values.

**Mode**: the value that appears most often.

**Standard deviation**: how spread out the values are from the mean. The smaller the deviation, the closer the values are to the mean. 

**Variance**: the standard deviation squared. It measures the same thing as standard deviation, but in a different way (in squared units, which is why standard deviation is easier to read).